In [1]:
import numpy as np
import plotly.graph_objects as go

def compute_surface(w, resolution=60):
    r = np.linspace(0, 0.999, resolution)
    theta = np.linspace(0, 2 * np.pi, resolution)
    r_grid, theta_grid = np.meshgrid(r, theta)
    x = r_grid * np.cos(theta_grid)
    y = r_grid * np.sin(theta_grid)
    z = np.sqrt(np.maximum(0.0, 1.0 - x**2 - y**2)) / w
    return x, y, z

def get_traces_for_state(w, a, b=0.0):
    # Clamp inside unit disc
    if a**2 + b**2 >= 0.99:
        scale = np.sqrt(0.98 / (a**2 + b**2))
        a *= scale
        b *= scale

    # Surface
    x_s, y_s, z_s = compute_surface(w)
    surface = go.Surface(x=x_s, y=y_s, z=z_s, colorscale='Reds', opacity=0.7, showscale=False)

    # Point
    val = max(1e-6, 1.0 - a**2 - b**2)
    p0 = [a, b, np.sqrt(val) / w]
    point = go.Scatter3d(x=[p0[0]], y=[p0[1]], z=[p0[2]], mode='markers', marker=dict(size=6, color='forestgreen'))

    # Expr 3 Normal
    fa = -a / (w * np.sqrt(val))
    fb = -b / (w * np.sqrt(val))
    n3 = np.sqrt(1.0 + fa**2 + fb**2)
    dir3 = np.array([fa, fb, -1.0]) / n3
    p3 = np.array(p0) + dir3
    norm3 = go.Scatter3d(x=[p0[0], p3[0]], y=[p0[1], p3[1]], z=[p0[2], p3[2]], mode='lines', line=dict(color='green', width=5))

    # Expr 9 Scaled Normal
    eps = 1e-4
    f_aa = ((- (a + eps) / (w * np.sqrt(max(1e-6, 1.0 - (a + eps)**2 - b**2)))) -
            (- (a - eps) / (w * np.sqrt(max(1e-6, 1.0 - (a - eps)**2 - b**2))))) / (2 * eps)
    f_bb = ((- b / (w * np.sqrt(max(1e-6, 1.0 - a**2 - (b + eps)**2)))) -
            (- b / (w * np.sqrt(max(1e-6, 1.0 - a**2 - (b - eps)**2))))) / (2 * eps)
    v = f_aa / (f_bb**3) if abs(f_bb) > 1e-5 else 9.0
    ga = v * fa
    n9 = np.sqrt(1.0 + ga**2)
    p9 = np.array([a, 0.0, p0[2]]) + np.array([ga / n9, 0.0, -1.0 / n9])
    norm9 = go.Scatter3d(x=[a, p9[0]], y=[0.0, p9[1]], z=[p0[2], p9[2]], mode='lines', line=dict(color='darkorange', width=5))

    return [surface, point, norm3, norm9]

# Precompute slider frames for 'a' parameter (with fixed w=3)
a_values = np.linspace(-0.85, 0.85, 25)
initial_traces = get_traces_for_state(w=3.0, a=0.42)

fig = go.Figure(data=initial_traces)

frames = []
for a_val in a_values:
    frames.append(go.Frame(data=get_traces_for_state(w=3.0, a=a_val), name=f"{a_val:.2f}"))

fig.frames = frames

fig.update_layout(
    title="Spheroid Normals Visualizer (Native Plotly Slider)",
    scene=dict(
        xaxis=dict(range=[-1.2, 1.2]),
        yaxis=dict(range=[-1.2, 1.2]),
        zaxis=dict(range=[-0.8, 0.8]),
        aspectratio=dict(x=1, y=1, z=0.8)
    ),
    sliders=[{
        "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                   "label": f"{float(f.name):.2f}", "method": "animate"} for f in frames],
        "currentvalue": {"prefix": "Parameter a: "},
        "pad": {"t": 30}
    }]
)

fig.show()